# Dataset Pipeline - Local Processing

**Purpose**: Process vulnerability datasets locally from `raw/` directory and export standardized JSONL files to `datasets/` folder.

**Prerequisites**: Run `00_setup.ipynb` first

**Output**: Three splits (train/val/test) with identical structure:

```json
{
  "lines": [normalized_code_lines],
  "raw_lines": [original_code_lines],
  "label": [0/1_vulnerability_labels],
  "type": [statement_types],
  "cwe_id": "CWE-XXX",
  "dataset_source": "source_name"
}
```


## Section 1: Imports & Configuration


In [ ]:
import os
import json
import re
import ast
import textwrap
import difflib
import shutil
import random
from collections import Counter
from pathlib import Path

In [ ]:
# Configure paths (LOCAL)
BASE_DIR = Path("/Users/anas/Projects/code-security-identifier")
RAW_DIR = BASE_DIR / "raw"
DATASETS_DIR = BASE_DIR / "datasets"

# Create directories
DATASETS_DIR.mkdir(parents=True, exist_ok=True)

random.seed(42)

print(f"✓ Base directory: {BASE_DIR}")
print(f"✓ Raw data directory: {RAW_DIR}")
print(f"✓ Output directory: {DATASETS_DIR}")

## Section 2: Helper Functions


In [ ]:
# Statement type mapping for AST analysis
STMT_TYPES = {
    ast.Import: "Import", ast.ImportFrom: "Import From",
    ast.Assign: "Assign", ast.AugAssign: "Augmented assign",
    ast.AnnAssign: "Assign", ast.Return: "Return",
    ast.If: "Condition", ast.For: "For/While", ast.While: "For/While",
    ast.With: "Expression", ast.Assert: "Assert", ast.Expr: "Expression",
    ast.FunctionDef: "Function declaration",
    ast.AsyncFunctionDef: "Function declaration",
    ast.ClassDef: "Expression", ast.Try: "Expression",
    ast.Raise: "Expression", ast.Pass: "Expression",
    ast.Break: "Expression", ast.Continue: "Expression",
}

def split_into_statements(code_str):
    """
    Parse code into statements with types.
    Returns: (raw_lines, normalized_lines, types) or None if unparseable.
    """
    if not code_str or not code_str.strip():
        return None

    # Get statement types via AST
    line_types = {}
    try:
        tree = ast.parse(textwrap.dedent(code_str))
        for node in ast.walk(tree):
            if hasattr(node, "lineno") and isinstance(node, ast.stmt):
                line_types[node.lineno] = STMT_TYPES.get(type(node), "Expression")
    except SyntaxError:
        # Try wrapping in function
        try:
            wrapped = "def __w__():\n" + textwrap.indent(textwrap.dedent(code_str), "    ")
            tree = ast.parse(wrapped)
            for node in ast.walk(tree):
                if hasattr(node, "lineno") and isinstance(node, ast.stmt):
                    line_types[node.lineno - 1] = STMT_TYPES.get(type(node), "Expression")
        except SyntaxError:
            pass

    dedented = textwrap.dedent(code_str)
    all_lines = dedented.split("\n")

    raw, norm, types = [], [], []
    for i, line in enumerate(all_lines):
        if not line.strip():
            continue
        raw.append(line)
        norm.append(" ".join(line.split()))

        s = line.strip()
        if s.startswith('"""') or s.startswith("'''"):
            types.append("Docstring")
        else:
            types.append(line_types.get(i + 1, "Expression"))

    return (raw, norm, types) if raw else None

def label_vulnerable_lines(raw_lines, bad_texts):
    """
    Label lines as vulnerable (1) or safe (0) based on bad code patterns.
    """
    labels = [0] * len(raw_lines)
    if not bad_texts:
        return labels

    for i, line in enumerate(raw_lines):
        lc = line.strip()
        if not lc:
            continue
        for bt in bad_texts:
            bc = bt.strip()
            if not bc:
                continue
            if bc in lc or lc in bc or " ".join(bc.split()) == " ".join(lc.split()):
                labels[i] = 1
                break
    return labels

def write_jsonl(path, records):
    """
    Write list of records to JSONL file.
    """
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        for r in records:
            f.write(json.dumps(r) + "\n")

def read_jsonl(path):
    """
    Read JSONL file into list of records.
    """
    if not os.path.exists(path):
        return []
    with open(path) as f:
        return [json.loads(line) for line in f]

print("✓ Helper functions loaded")

## Section 3: Process Local Vudenc Dataset


In [ ]:
# CWE mappings
CWE = {
    "command_injection": "CWE-077", "open_redirect": "CWE-601",
    "path_disclosure": "CWE-022", "remote_code_execution": "CWE-094",
    "sql": "CWE-089", "xsrf": "CWE-352", "xss": "CWE-079",
}

vudenc_records = []
skipped = 0

print("Processing local Vudenc dataset...")
for vuln_name, cwe_id in CWE.items():
    path = RAW_DIR / f"plain_{vuln_name}"
    if not path.exists():
        print(f"  ⊘ {vuln_name}: file not found")
        continue

    with open(path) as f:
        data = json.load(f)

    count = 0
    for repo, commits in data.items():
        for sha, info in commits.items():
            for fpath, fdata in info.get("files", {}).items():
                source = fdata.get("sourceWithComments", "") or fdata.get("source", "")
                changes = fdata.get("changes", [])

                bads = []
                for ch in changes:
                    if isinstance(ch, dict):
                        bads.extend(ch.get("badparts", []))

                if not source or not bads:
                    continue

                result = split_into_statements(source)
                if result is None:
                    skipped += 1
                    continue

                raw, norm, types = result
                labels = label_vulnerable_lines(raw, bads)

                if sum(labels) == 0:
                    skipped += 1
                    continue

                vudenc_records.append({
                    "lines": norm, "raw_lines": raw,
                    "label": labels, "type": types,
                    "cwe_id": cwe_id, "dataset_source": "vudenc",
                })
                count += 1

    print(f"  ✓ {vuln_name} ({cwe_id}): {count:,} functions")

print(f"✓ Local Vudenc: {len(vudenc_records):,} functions ({skipped:,} skipped)")

## Section 4: Process Function-Level Dataset


In [ ]:
func_records = []
skipped = 0

path = RAW_DIR / "function_level_dataset.out"
print("Processing function-level dataset...")

if not path.exists():
    print(f"  ⊘ function_level_dataset.out not found")
else:
    with open(path) as f:
        records = [json.loads(l) for l in f]

    for rec in records:
        if rec.get("programming_language") != "Python":
            continue

        before = rec.get("code_before", "")
        after = rec.get("code_after", "")

        if not before.strip():
            continue

        result = split_into_statements(before)
        if result is None:
            skipped += 1
            continue

        raw, norm, types = result

        # Diff before/after to find changed lines
        if after.strip():
            removed = set()
            for d in difflib.unified_diff(before.split("\n"), after.split("\n"), lineterm=""):
                if d.startswith("-") and not d.startswith("---"):
                    removed.add(d[1:].strip())

            labels = [0] * len(raw)
            for i, line in enumerate(raw):
                if line.strip() in removed:
                    labels[i] = 1
        else:
            labels = [1] * len(raw)

        # Extract CWE from description
        gpt = rec.get("gpt_answer", "") + " " + rec.get("description", "")
        m = re.search(r"CWE-\d+", gpt)
        cwe_id = m.group(0) if m else "unknown"

        func_records.append({
            "lines": norm, "raw_lines": raw,
            "label": labels, "type": types,
            "cwe_id": cwe_id, "dataset_source": "funclevel",
        })

        # Also add the fixed (safe) version
        if after.strip():
            r2 = split_into_statements(after)
            if r2:
                raw2, norm2, types2 = r2
                func_records.append({
                    "lines": norm2, "raw_lines": raw2,
                    "label": [0] * len(raw2), "type": types2,
                    "cwe_id": cwe_id, "dataset_source": "funclevel",
                })

print(f"✓ Function-level: {len(func_records):,} functions ({skipped:,} skipped)")

## Section 5: Process Security Eval Dataset


In [ ]:
sec_records = []

path = RAW_DIR / "dataset.jsonl"
print("Processing security eval dataset...")

if not path.exists():
    print(f"  ⊘ dataset.jsonl not found")
else:
    with open(path) as f:
        for line in f:
            rec = json.loads(line)
            cwe_id = rec["ID"].split("_")[0]

            result = split_into_statements(rec["Insecure_code"])
            if result is None:
                continue

            raw, norm, types = result
            sec_records.append({
                "lines": norm, "raw_lines": raw,
                "label": [1] * len(raw), "type": types,
                "cwe_id": cwe_id, "dataset_source": "securityeval",
            })

    cwe_count = len(set(r['cwe_id'] for r in sec_records))
    print(f"✓ Security Eval: {len(sec_records):,} functions, {cwe_count} unique CWE types")

## Section 6: Combine & Split into Train/Val/Test


In [ ]:
print("Combining all datasets...")

# Combine all data
train_all = vudenc_records + func_records + sec_records

print(f"  Total records: {len(train_all):,}")

# Shuffle and split into train/val
random.shuffle(train_all)
val_size = max(1, len(train_all) // 10)  # 10% validation
final_val = train_all[:val_size]
final_train = train_all[val_size:]

print(f"  Train: {len(final_train):,}")
print(f"  Val:   {len(final_val):,}")

# Save to datasets folder
print("\nSaving datasets...")
for name, data in [("FINAL_train", final_train), ("FINAL_val", final_val)]:
    path = DATASETS_DIR / f"{name}.jsonl"
    write_jsonl(path, data)

    n = len(data)
    stmts = sum(len(r["label"]) for r in data)
    vuln = sum(sum(r["label"]) for r in data)
    pct = vuln / stmts * 100 if stmts > 0 else 0
    print(f"  ✓ {name}: {n:,} functions, {stmts:,} statements ({vuln:,} vulnerable, {pct:.1f}%)")

print(f"\n✓ Datasets ready at: {DATASETS_DIR}")

## Section 7: Cleanup - Delete Raw Directory


In [ ]:
print("Cleanup: Removing raw directory...\n")

if RAW_DIR.exists():
    shutil.rmtree(RAW_DIR)
    print(f"✓ Deleted {RAW_DIR}")
else:
    print(f"⊘ {RAW_DIR} not found")

print("\n✓ Cleanup complete")